# 02-1. Mutation Burden Proxy Features

각 샘플에 대해 mutation burden proxy feature를 계산하여 train.csv에 붙인 뒤 CSV로 저장한다.

| Feature | 정의 |
|---|---|
| `mutated_gene_count` | 변이가 존재하는 고유 유전자 수 |
| `mutation_event_count` | 공백으로 분리한 전체 mutation token 수 |
| `synonymous_event_count` | 동의 변이 token 수 (예: R895R) |
| `functional_event_count` | 단백질 변화를 유발하는 변이 token 수 |
| `missense_event_count` | 아미노산 치환 변이 token 수 (예: R132H) |
| `nonsense_event_count` | stop codon 생성 변이 token 수 (예: R213*) |
| `frameshift_event_count` | 읽기 틀 변화 변이 token 수 (예: K16fs) |
| `complex_event_count` | 구조적 복합 변이 token 수 (예: E746_A750del) |
| `multihit_gene_count` | 한 유전자에 2개 이상 변이가 있는 유전자 수 |
| `max_events_per_gene` | 한 유전자에서 관찰된 최대 mutation token 수 |
| `no_mutation_flag` | 전체 유전자에 변이가 없으면 1 |
| `mutation_gene_ratio` | 변이가 존재하는 유전자 수 / 전체 유전자 수 |
| `multi_mutation_gene_ratio` | 복수 변이 유전자 수 / 변이가 존재하는 유전자 수 |
| `n_unique_mutation_tokens` | 중복을 제거한 mutation token 수 |
| `n_repeated_mutation_tokens` | 중복 등장 mutation token 수 (전체 - 고유) |

In [1]:
import sys
from pathlib import Path

ROOT = Path("__file__").resolve().parents[1]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_RAW = ROOT / "data" / "raw"
DATA_OUT = ROOT / "data" / "process"
DATA_OUT.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
from cancer_hack.parser import compute_all_burden_features


## 1. 데이터 로드

In [3]:
train = pd.read_csv(DATA_RAW / "train.csv")
gene_cols = [c for c in train.columns if c not in {"ID", "SUBCLASS"}]

print(f"shape     : {train.shape}")
print(f"gene 수   : {len(gene_cols)}")
train.head(3)

shape     : (6201, 4386)
gene 수   : 4384


,ID,SUBCLASS,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TRAIN_0000,KIPAN,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
1,TRAIN_0001,SARC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
2,TRAIN_0002,SKCM,R895R,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT


## 2. Feature 계산

In [4]:
features = compute_all_burden_features(train, gene_cols)

print(f"features shape: {features.shape}")
features.head(3)


features shape: (6201, 18)


,mutated_gene_count,mutation_event_count,synonymous_event_count,functional_event_count,missense_event_count,nonsense_event_count,frameshift_event_count,complex_event_count,multihit_gene_count,max_events_per_gene,no_mutation_flag,mutation_gene_ratio,multi_mutation_gene_ratio,n_unique_mutation_tokens,n_repeated_mutation_tokens,explicit_deletion_event_count,explicit_deletion_gene_count,has_explicit_deletion
0,25,25,7,18,17,1,0,0,0,1,0,0.005703,0.000000,25,0,0,0,0
1,17,17,5,12,9,0,3,0,0,1,0,0.003878,0.000000,17,0,0,0,0
2,119,124,39,85,79,6,0,0,5,2,0,0.027144,0.042017,124,0,0,0,0


## 3. 기술 통계 확인

In [5]:
features.describe().round(4)

,mutated_gene_count,mutation_event_count,synonymous_event_count,functional_event_count,missense_event_count,nonsense_event_count,frameshift_event_count,complex_event_count,multihit_gene_count,max_events_per_gene,no_mutation_flag,mutation_gene_ratio,multi_mutation_gene_ratio,n_unique_mutation_tokens,n_repeated_mutation_tokens,explicit_deletion_event_count,explicit_deletion_gene_count,has_explicit_deletion
count,6201.0000,6201.0000,6201.0000,6201.000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000,6201.0000
mean,35.2996,41.1488,10.7858,30.363,26.5667,2.1590,1.5981,0.0392,3.5520,1.5348,0.0152,0.0081,0.0354,39.8649,1.2840,0.0006,0.0006,0.0006
std,100.7764,189.8051,58.7292,133.109,121.6129,11.3151,5.8957,0.2872,38.1773,1.4926,0.1222,0.0230,0.1084,150.1740,68.4455,0.0254,0.0254,0.0254
min,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,7.0000,7.0000,1.0000,5.000,4.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0016,0.0000,7.0000,0.0000,0.0000,0.0000,0.0000
50%,14.0000,14.0000,3.0000,11.000,9.0000,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0032,0.0000,14.0000,0.0000,0.0000,0.0000,0.0000
75%,27.0000,28.0000,7.0000,21.000,19.0000,2.0000,2.0000,0.0000,1.0000,2.0000,0.0000,0.0062,0.0385,28.0000,0.0000,0.0000,0.0000,0.0000
max,2393.0000,9972.0000,3616.0000,6356.000,6002.0000,335.0000,139.0000,12.0000,2021.0000,64.0000,1.0000,0.5458,1.0000,4593.0000,5379.0000,1.0000,1.0000,1.0000


## 4. 원본에 병합 후 CSV 저장

In [ ]:
out = pd.concat([train[["ID", "SUBCLASS"]], features], axis=1)

out_path = DATA_OUT / "train_mutation_burden.csv"
out.to_csv(out_path, index=False)

print(f"저장 완료: {out_path}")
print(f"shape    : {out.shape}")
out.head(3)